
# Creazione dei tensori per il modello surrogato IRIDE

Questo notebook esegue il calcolo delle dosi assorbite dagli organi oculari (e dal tumore) a partire dai file di simulazione Monte Carlo / Geant4 per diverse posizioni angolari della placca brachiterapica.

Nel contesto del progetto IRIDE, l'obiettivo fisico-computazionale è utilizzare distribuzioni di dose ottenute con simulazioni Monte Carlo per addestrare un modello surrogato differenziabile del piano di trattamento. Il modello surrogato deve poi essere utilizzabile nella fase successiva di ottimizzazione della posizione della placca.

**Obiettivi principali:**
1. **Configurazione parametri e costanti fisiche/geometriche.**
2. **Localizzazione del picco del tumore (apice)** e definizione della regione apicale.
3. **Ciclo sugli angoli di rotazione ($\theta$):**
   - Calcolo delle feature geometriche di input ($[\sin\theta, \cos\theta, \sin\phi, \cos\phi, r]$).
   - Calcolo del tempo necessario a erogare la dose bersaglio ($50\text{ Gy}$) nella regione apicale del tumore.
   - Calcolo della dose media, deviazione standard e dose massima per ciascun organo di interesse.
4. **Esportazione dei tensori NumPy:**
   - `input_tensor.npy` — Feature geometriche di input ($\sin\theta, \cos\theta, \sin\phi, \cos\phi, r$) [Dimensione: $N \times 5$]
   - `output_tensor.npy` — Metriche dosimetriche e temporali (Tempo di trattamento + 6 organi $\times$ 3 metriche: media, std, max) [Dimensione: $N \times 19$]

In [ ]:
import numpy as np
import pandas as pd
import math
import os
import re
import sys
from pathlib import Path

: 

## Configurazione di Percorsi, Costanti e Strutture Dati

In questa cella vengono definiti i percorsi di input e output e le costanti globali usate nel resto del codice.

`data_dir` è la cartella che contiene i file `.txt` prodotti dalle simulazioni Monte Carlo della brachiterapia oculare.

Ogni file rappresenta una specifica configurazione della placca ed è identificato da:
- `theta`: angolo polare della placca rispetto all'occhio;
- `phi`: angolo azimutale (in questo studio fissato a 0°);
- `s`: indice del run statistico Monte Carlo (da 1 a 10).

I file seguono la convenzione di nomenclatura:
`dose_voxel_theta{theta}_phi{phi}_s{s}.txt`

La prima riga del file contiene la posizione della placca, mentre le righe successive riportano le informazioni voxel-per-voxel della distribuzione di dose:

| Colonna | Significato |
| :--- | :--- |
| **voxelId** | Identificativo univoco del voxel |
| **organId** | Identificativo dell'organo a cui appartiene il voxel |
| **x_mm** | Coordinata x del centro del voxel [mm] |
| **y_mm** | Coordinata y del centro del voxel [mm] |
| **z_mm** | Coordinata z del centro del voxel [mm] |
| **dose_Gy_per_event** | Dose depositata per decadimento simulato [Gy/evento] |
| **n_contributi** | Numero di eventi che hanno contribuito alla dose del voxel |

Nel progetto sono disponibili 72 configurazioni angolari della placca ($\theta$ da $0^\circ$ a $355^\circ$ con passo di $5^\circ$) e, per ciascuna configurazione, 10 run Monte Carlo indipendenti utilizzate per stimare la variabilità statistica della simulazione.

`out_dir` è la cartella in cui verranno salvati i tensori finali (`input_tensor.npy` e `output_tensor.npy`).



### Contestualizzazione Fisica e Statistica delle Simulazioni

Le simulazioni Monte Carlo modellate in questo progetto riproducono il trattamento di retinoblastoma tramite una placca brachiterapica a **Rutenio-106 ($^{106}\text{Ru}$)**, un emettitore $\beta^-$. 

Il $^{106}\text{Ru}$ decade in $^{106}\text{Rh}$, il quale decade a sua volta in $^{106}\text{Pd}$ emettendo elettroni $\beta$ ad alta energia responsabili del deposito di dose nei tessuti oculari. Il corto range degli elettroni garantisce una forte concentrazione della dose in prossimità della placca, risparmiando i tessuti più distanti.

#### 1. Conversione da Simulazione a Rateo di Dose Clinico
Nelle simulazioni Monte Carlo la dose viene calcolata per singolo evento primario (`dose_Gy_per_event`). Per ricondurre il dato a una grandezza clinicamente rilevante, si assume un'attività della sorgente di **$10\text{ MBq}$** ($10^7\text{ Bq}$), in accordo con le specifiche della Task 1 del progetto IRIDE.

Il rateo di dose fisico in Gy/s viene quindi calcolato come:
$$\text{dose\_rate\_Gy\_s} = \text{dose\_Gy\_per\_event} \times \text{ATTIVITA\_BQ}$$

Questo valore viene successivamente convertito in $\text{Gy/giorno}$ tramite il fattore di conversione temporale ($3600 \times 24$).

#### 2. Significato Statistico delle Run Monte Carlo
Per ciascuna delle 72 configurazioni angolari della placca sono disponibili **10 run Monte Carlo indipendenti** (`N_RUN = 10`), ciascuna basata su $10^6$ eventi primari (per un totale di $10^7$ decadimenti campionati per angolo). 

Le 10 run rappresentano realizzazioni indipendenti dello stesso esperimento e consentono di:
* **Ridurre la varianza** mediante l'aggregazione media dei risultati.
* **Quantificare l'incertezza statistica** del campionamento Monte Carlo.
* **Garantire la robustezza del dataset** destinato all'addestramento del modello surrogato differenziabile.

In [ ]:
# ==========================================
# 1. PERCORSI E DIRECTORY
# ==========================================
data_dir = Path("/Users/simonenardi/Desktop/Dose")
out_dir = Path("/Users/simonenardi/Desktop/IRIDE/Risultati")

out_dir.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. COSTANTI GEOMETRICHE E FISICHE
# ==========================================
phi_fisso = 0.0          # Angolo phi di riferimento
raggio_placca = 12.1     # Raggio della placca applicatrice (mm)
ATTIVITA_BQ = 10e6       # Attività della sorgente (10 MBq in Bq)
N_RUN = 10               # Numero di run per ciascuna configurazione angolare

# Costanti per Apice e Tumore
ALTEZZA_APICE = 6.0      # Altezza clinica dell'apice traslato (mm)
RAGGIO_SFERA = 3.5       # Raggio sfera per la regione apicale (mm)
VOXEL_PITCH = 0.5        # Passo della griglia voxel (mm)
CENTRO_OCCHIO = np.array([0.0, 0.0, 0.0])

# Costanti per Dosimetria
DOSE_BERSAGLIO_TUMORE = 50.0   # Dose target (Gy)
FATTORE_CONVERSIONE = 3600.0 * 24  # Conversione da Gy/s a Gy/giorno

# ==========================================
# 3. MAPPATURA E ORDINE DEGLI ORGANI
# ==========================================
ORGANI = {
    0: "Aria", 
    1415: "Sclera", 
    1417: "Cornea", 
    1418: "Retina",
    1420: "Cristallino", 
    2882: "Nervo Ottico", 
    9000: "Umor Vitreo", 
    9999: "Tumore",
}
TUMOR_ID = 9999

ORGANI_STAMPA = ["Cristallino", "Retina", "Cornea", "Nervo Ottico", "Sclera", "Tumore"]
ID_ORGANI_STAMPA = [1420, 1418, 1417, 2882, 1415, 9999]
ORDINE_ORGANI_OUTPUT = [1420, 1418, 1417, 2882, 1415, 9999]

# Inizializzazione contenitori per i tensori
input_features = []
output_features = []

---
## 2. Identificazione e Traslazione della Regione Apicale

In questa sezione viene elaborata la simulazione di riferimento ($\theta = 0^\circ, \phi = 0^\circ, \text{run}=1$) per definire la geometria e la posizione della **regione apicale del tumore**.

#### Motivazione Fisico-Clinica e Sequenza Geometrica:
La geometria tumorale presente nel modello di simulazione originale risultava di dimensioni maggiori rispetto a quella tipicamente trattata in ambiente clinico. Di conseguenza, la dose calcolata sul picco tumorale originale portava a tempi di trattamento irrealistici e non coerenti con la pratica clinica (molto superiori alla finestra temporale standard di **2-7 giorni**).

Per ripristinare la consistenza con la fisica e la clinica reale, il codice esegue i seguenti passaggi in sequenza:

1. **Identificazione dell'Apice Reale:** Tra tutti i voxel del tumore (`organId = 9999`), si trova il voxel a massima profondità ($R_{\text{placca}} - x$).
2. **Definizione della Regione Apicale Reale:** Attorno a questo apice originale si isola una regione sferica di raggio $R = 3.5\text{ mm}$.
3. **Traslazione e Mappatura:** Si calcola l'offset necessario per portare l'apice all'altezza clinica target ($\text{ALTEZZA\_APICE} = 6.0\text{ mm}$). Le coordinate della regione sferica vengono traslate di tale offset e rimappate sulla griglia dei voxel tumorali.

Gli ID dei voxel appartenenti a questa **regione apicale traslata** (`lista_voxel_traslati`) costituiscono il volume di riferimento fisso: per ogni posizione angolare della placca, il tempo di trattamento verrà calcolato come il tempo necessario ad erogare la dose target di $50\text{ Gy}$ a questo specifico gruppo di voxel.

In [ ]:
# ==========================================
# 2. CALCOLO APICE E TRASLAZIONE (Setup Iniziale)
# ==========================================
theta_apice = 0
apice_file_s1 = f"dose_voxel_theta{theta_apice}_phi{int(phi_fisso)}_s1.txt"
percorso_file_s1 = data_dir / apice_file_s1

# Verifica presenza file di riferimento
if not percorso_file_s1.exists():
    print(f"Errore critico: file di riferimento non trovato in {percorso_file_s1}")
    sys.exit()

# Caricamento dati voxel del file di riferimento
df_rif = pd.read_csv(
    percorso_file_s1, sep='\t', comment='#',
    names=['voxelId', 'organId', 'x_mm', 'y_mm', 'z_mm', 'dose_Gy_per_nEv', 'n_contributi']
)

# Filtraggio voxel tumorali e calcolo profondità
tumor_rif = df_rif[df_rif['organId'] == TUMOR_ID].copy()
tumor_rif['depth_mm'] = raggio_placca - tumor_rif['x_mm']
print(f"Voxel tumorali totali (riferimento): {len(tumor_rif)}")

if tumor_rif.empty:
    print("Errore critico: Nessun voxel tumorale trovato nel file di riferimento.")
    sys.exit()

# 1. Identificazione dell'apice (voxel a massima profondità)
idx_apice      = tumor_rif['depth_mm'].idxmax()
apice          = tumor_rif.loc[idx_apice]

APICE_VOXEL_ID = int(apice['voxelId'])
APICE_X, APICE_Y, APICE_Z = apice['x_mm'], apice['y_mm'], apice['z_mm']
APICE_DEPTH    = apice['depth_mm']

# 2. Calcolo offset e coordinate traslate
OFFSET         = APICE_DEPTH - ALTEZZA_APICE
APICE_X_TR     = APICE_X + OFFSET
APICE_DEPTH_TR = raggio_placca - APICE_X_TR

# 3. Regione apicale reale (sfera R = 3.5 mm attorno all'apice reale)
d_reale = np.sqrt((tumor_rif['x_mm'] - APICE_X)**2 + (tumor_rif['y_mm'] - APICE_Y)**2 + (tumor_rif['z_mm'] - APICE_Z)**2)
regione_reale = tumor_rif[d_reale <= RAGGIO_SFERA].copy()

# 4. Mappatura della regione apicale sulle coordinate traslate
coords_tumore = tumor_rif[['x_mm', 'y_mm', 'z_mm']].values
righe_traslate = []

for _, vox in regione_reale.iterrows():
    xt, yt, zt = vox['x_mm'] + OFFSET, vox['y_mm'], vox['z_mm']
    distanze = np.sqrt((coords_tumore[:, 0] - xt)**2 + (coords_tumore[:, 1] - yt)**2 + (coords_tumore[:, 2] - zt)**2)
    idx_min = np.argmin(distanze)
    if distanze[idx_min] <= VOXEL_PITCH:
        righe_traslate.append(tumor_rif.iloc[idx_min])
        
regione_traslata = pd.DataFrame(righe_traslate).drop_duplicates(subset='voxelId').copy()
lista_voxel_traslati = list(set(regione_traslata['voxelId'].values))

print(f"Apice identificato - Voxel ID: {APICE_VOXEL_ID} | Profondità Reale: {APICE_DEPTH:.2f} mm -> Traslata: {APICE_DEPTH_TR:.2f} mm")
print(f"Regione apicale identificata: {len(lista_voxel_traslati)} voxel.")
print("="*90)

Voxel tumorali totali: 5916


---
## 3. Ciclo di Elaborazione Dosimetrica sui 72 Angoli ($\theta$)

In questa sezione si cicla su tutte le 72 configurazioni angolari della placca ($\theta \in [0^\circ, 355^\circ]$ a passi di $5^\circ$). 

Per ciascun angolo, il codice esegue i seguenti passaggi:

1. **Estrazione Feature di Input:** 
   Estrae la posizione della placca dalla prima riga del file, calcola il raggio totale $r$ e converte gli angoli in coordinate trigonometriche: $[\sin\theta, \cos\theta, \sin\phi, \cos\phi, r]$.
2. **Aggregazione delle Run Monte Carlo:** 
   Carica i file relativi alle 10 run indipendenti per l'angolo corrente e converte la dose per evento nel rateo di dose fisico ($\text{Gy/s}$) moltiplicando per l'attività della sorgente ($10\text{ MBq}$).
3. **Calcolo del Tempo di Trattamento:** 
   Calcola il rateo di dose medio erogato alla **regione apicale traslata** (i voxel identificati nella Sezione 2) e determina il tempo (in giorni) necessario per raggiungere la dose target di $50\text{ Gy}$.
4. **Calcolo Dosi agli Organi e al Tumore:** 
   Utilizzando il tempo di trattamento calcolato, determina per ciascun organo di rischio (OAR) e per il tumore:
   - **Dose Media (Gy)** e la sua **Deviazione Standard** tra le 10 run;
   - **Dose Massima (Gy)** assorbita dal voxel più caldo (*hotspot*) dell'organo.

In [ ]:
# ==========================================
# 3. CICLO SUI 72 ANGOLI
# ==========================================
for theta in range(0, 360, 5):
    # Gestione convenzione nomi file per angoli >= 320°
    theta_file = theta - 360 if theta >= 320 else theta

    # -- 3.1 Costruzione features di Input --
    nome_file_s1 = f"dose_voxel_theta{theta_file}_phi{int(phi_fisso)}_s1.txt"
    percorso_file_s1 = data_dir / nome_file_s1

    if not percorso_file_s1.exists():
        print(f"[Attenzione] File non trovato per theta={theta}°. Angolo saltato.")
        continue

    # Lettura dell'intestazione per estrarre la posizione geometrica
    with open(percorso_file_s1, 'r') as f:
        prima_riga = f.readline()
        
    start_idx, end_idx = prima_riga.find('('), prima_riga.find(')')
    riga_input_valida = False

    if start_idx != -1 and end_idx != -1:
        try:
            x, y, z = map(float, prima_riga[start_idx+1 : end_idx].split(','))
            raggio_totale = np.sqrt(x**2 + y**2 + z**2) + raggio_placca
            theta_rad, phi_rad = np.radians(theta), np.radians(phi_fisso)
            
            # Vettore di input temporaneo per l'angolo corrente
            current_input = [np.sin(theta_rad), np.cos(theta_rad), np.sin(phi_rad), np.cos(phi_rad), raggio_totale]
            riga_input_valida = True
        except ValueError:
            pass

    if not riga_input_valida:
        print(f"[Attenzione] Header non valido per theta={theta}°. Angolo saltato.")
        continue

    # -- 3.2 Caricamento di tutte le Run del dato angolo --
    runs = []
    for i in range(1, N_RUN + 1):
        filepath = data_dir / f"dose_voxel_theta{theta_file}_phi{int(phi_fisso)}_s{i}.txt"
        if filepath.exists():
            df = pd.read_csv(filepath, sep='\t', comment='#', names=['voxelId', 'organId', 'x_mm', 'y_mm', 'z_mm', 'dose_Gy_per_event', '_extra']).drop(columns=['_extra'], errors='ignore')
            df["run"] = i
            runs.append(df)

    if not runs:
        print(f"[Attenzione] Nessuna run caricata per theta={theta}°. Angolo saltato.")
        continue

    # Unione delle run in un unico DataFrame e conversione Dose Rate in Gy/s
    df_all = pd.concat(runs, ignore_index=True)
    df_all["dose_rate_Gy_s"] = df_all["dose_Gy_per_event"] * ATTIVITA_BQ

    # -- 3.3 Calcolo Tempo su Regione Apicale --
    df_apice = df_all[df_all["voxelId"].isin(lista_voxel_traslati)]
    if df_apice.empty:
        print(f"[Attenzione] Angolo {theta}°: Nessun voxel della regione apicale trovato. Angolo saltato.")
        continue

    ratei_apice_runs = df_apice.groupby("run")["dose_rate_Gy_s"].mean()
    rateo_apice_medio_giorno = ratei_apice_runs.mean() * FATTORE_CONVERSIONE
    tempo_trattamento_giorni = DOSE_BERSAGLIO_TUMORE / rateo_apice_medio_giorno

    # -- 3.4 Dose Media e Massima agli Organi (Gy/s) --
    matrice_rate_organi = df_all.groupby(["organId", "run"])["dose_rate_Gy_s"].mean().unstack("run")
    rateo_medio_organi_s = matrice_rate_organi.mean(axis=1)
    
    # Voxel più caldo (Hotspot) per ogni organo integrato sulle N_RUN
    rateo_medio_voxel_s = df_all.groupby(["organId", "voxelId"])["dose_rate_Gy_s"].sum() / N_RUN
    rateo_max_organi_s = rateo_medio_voxel_s.groupby("organId").max()

    # -- 3.5 Stampe di Controllo --
    print(f"\n➔ Angolo Theta: {theta}° (File: theta{theta_file})")
    print(f"  • Tempo per {DOSE_BERSAGLIO_TUMORE} Gy all'apice: {tempo_trattamento_giorni:.4f} giorni ({tempo_trattamento_giorni*24:.2f} ore)")
    print(f"  • IMPATTO ORGANI (Integrato in {tempo_trattamento_giorni:.4f} gg):")
    print(f"    {'-'*85}")
    print(f"    {'ORGANO':<16} | {'RATEO MEDIO (Gy/gg)':<19} | {'DOSE MEDIA (Gy)':<17} | {'DOSE MAX (Gy)':<15}")
    print(f"    {'-'*85}")
    
    for org_name, org_id in zip(ORGANI_STAMPA, ID_ORGANI_STAMPA):
        r_medio_s = rateo_medio_organi_s.get(org_id, 0.0)
        r_max_s   = rateo_max_organi_s.get(org_id, 0.0)
        
        r_medio_giorno = r_medio_s * FATTORE_CONVERSIONE
        r_max_giorno   = r_max_s * FATTORE_CONVERSIONE
        
        dose_accumulata_media = r_medio_giorno * tempo_trattamento_giorni
        dose_accumulata_max   = r_max_giorno * tempo_trattamento_giorni
        
        print(f"    {org_name:<16} | {r_medio_giorno:<19.4f} | {dose_accumulata_media:<17.4f} | {dose_accumulata_max:.4f} Gy")
    print(f"    {'-'*85}")

    # -- 3.6 Popolamento dei Vettori di Input e Output --
    matrice_dose_organi = matrice_rate_organi * FATTORE_CONVERSIONE * tempo_trattamento_giorni
    riga_corrente_output = [tempo_trattamento_giorni]

    for org_id in ORDINE_ORGANI_OUTPUT:
        if org_id in matrice_dose_organi.index:
            media_dose = matrice_dose_organi.loc[org_id].mean()
            std_dose = matrice_dose_organi.loc[org_id].std()
            if pd.isna(std_dose):
                std_dose = 0.0
            
            max_dose = rateo_max_organi_s.get(org_id, 0.0) * FATTORE_CONVERSIONE * tempo_trattamento_giorni
        else:
            media_dose, std_dose, max_dose = 0.0, 0.0, 0.0
            
        riga_corrente_output.extend([media_dose, std_dose, max_dose])

    # Aggiunge le feature solo se tutte le fasi per l'angolo corrente sono andate a buon fine
    input_features.append(current_input)
    output_features.append(riga_corrente_output)

---
## 4. Salvataggio dei Tensori di Input e Output

Nell'ultima fase, le liste `input_features` e `output_features` vengono convertite in array NumPy bidimensionali e salvate nella cartella di destinazione (`out_dir`) nei file `input_tensor.npy` e `output_tensor.npy`.

#### Struttura dei Tensori Generati:

* **`input_tensor.npy`** (Dimensioni: $N_{\text{angoli\_validi}} \times 5$):
  * **Colonne 0–1:** $\sin(\theta), \cos(\theta)$
  * **Colonne 2–3:** $\sin(\phi), \cos(\phi)$
  * **Colonna 4:** Raggio totale $r$ [mm]

* **`output_tensor.npy`** (Dimensioni: $N_{\text{angoli\_validi}} \times 19$):
  * **Colonna 0:** Tempo di trattamento per erogare $50\text{ Gy}$ all'apice [giorni]
  * **Colonne 1–18:** Triplette $(\text{Media}, \text{Std}, \text{Max})$ della dose assorbita [Gy] per i 6 organi nell'ordine:
    1. Cristallino (ID 1420)
    2. Retina (ID 1418)
    3. Cornea (ID 1417)
    4. Nervo Ottico (ID 2882)
    5. Sclera (ID 1415)
    6. Tumore (ID 9999)

In [ ]:
# ==========================================
# 4. SALVATAGGIO TENSORI
# ==========================================

# Conversione delle liste in Array NumPy
input_tensor = np.array(input_features)
output_tensor = np.array(output_features)

# Salvataggio su disco in formato binario .npy
file_input_path = out_dir / "input_tensor.npy"
file_output_path = out_dir / "output_tensor.npy"

np.save(file_input_path, input_tensor)
np.save(file_output_path, output_tensor)

# Output di conferma e riepilogo
print("\n" + "=" * 90)
print("ELABORAZIONE E SALVATAGGIO COMPLETATI CON SUCCESSO!")
print("=" * 90)
print(f"Directory di destinazione: {out_dir}")
print(
    f"• input_tensor.npy  -> Shape: {input_tensor.shape}  (Angoli validi x Features Input)"
)
print(
    f"• output_tensor.npy -> Shape: {output_tensor.shape} (Angoli validi x Features Output)"
)
print("\nDettaglio colonne Output (19 totali):")
print("  └ [0]: Tempo di trattamento (giorni)")
print(
    f"  └ [1-18]: Triplette (Media, Std, Max) per organi ID {ORDINE_ORGANI_OUTPUT}"
)

Tensori salvati in: /Users/Utente/Desktop/data/Risultat_output_main
input_tensor:   (72, 5)  → (n_angoli × 5 features)
output_tensor:  (72, 19)  → (n_angoli × 19 features)

Struttura output_tensor:
  [0]      tempo_giorni
  [1–3]  Cristallino (media, std, max)
  [4–6]  Retina (media, std, max)
  [7–9]  Cornea (media, std, max)
  [10–12]  Nervo Ottico (media, std, max)
  [13–15]  Sclera (media, std, max)
  [16–18]  Tumore (media, std, max)
